In [6]:
from pathlib import Path
import sys
import random

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [8]:
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    ReadoutError,
)

# ============================================================
# EXPERIMENT 06A — NISQ NOISE MODELS
# ============================================================

P_1Q = 0.001
P_2Q = 0.01
P_READOUT = 0.02

# ------------------------------------------------------------
# Depolarizing noise
# ------------------------------------------------------------

depolarizing_noise = NoiseModel()

error_1q = depolarizing_error(
    P_1Q,
    1
)

error_2q = depolarizing_error(
    P_2Q,
    2
)

# Actual transpiled 1-qubit gates:
# p, ry, h
depolarizing_noise.add_all_qubit_quantum_error(
    error_1q,
    ["p", "ry", "h"]
)

# Actual transpiled 2-qubit gate:
# cx
depolarizing_noise.add_all_qubit_quantum_error(
    error_2q,
    ["cx"]
)

# ------------------------------------------------------------
# Readout noise
# ------------------------------------------------------------

readout_noise = NoiseModel()

readout_error = ReadoutError([
    [1 - P_READOUT, P_READOUT],
    [P_READOUT, 1 - P_READOUT],
])

readout_noise.add_all_qubit_readout_error(
    readout_error
)

# ------------------------------------------------------------
# Combined noise
# ------------------------------------------------------------

combined_noise = NoiseModel()

combined_noise.add_all_qubit_quantum_error(
    error_1q,
    ["p", "ry", "h"]
)

combined_noise.add_all_qubit_quantum_error(
    error_2q,
    ["cx"]
)

combined_noise.add_all_qubit_readout_error(
    readout_error
)

print("Noise models created successfully.")

print("\nDepolarizing:")
print(depolarizing_noise)

print("\nReadout:")
print(readout_noise)

print("\nCombined:")
print(combined_noise)

Noise models created successfully.

Depolarizing:
NoiseModel:
  Basis gates: ['cx', 'h', 'id', 'p', 'ry', 'rz', 'sx']
  Instructions with noise: ['h', 'p', 'ry', 'cx']
  All-qubits errors: ['p', 'ry', 'h', 'cx']

Readout:
NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx']
  Instructions with noise: ['measure']
  All-qubits errors: ['measure']

Combined:
NoiseModel:
  Basis gates: ['cx', 'h', 'id', 'p', 'ry', 'rz', 'sx']
  Instructions with noise: ['p', 'h', 'cx', 'measure', 'ry']
  All-qubits errors: ['p', 'ry', 'h', 'cx', 'measure']


In [9]:
from qiskit import QuantumCircuit

qc_measure = qc_transpiled.copy()

qc_measure.measure_all()

print(qc_measure)
print("\nGate counts:")
print(qc_measure.count_ops())

        ┌───┐┌───────────┐                                             »
   q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■────■──»
        ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐  │  »
   q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──┼──»
        ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
   q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
        ├───┤├───────────┤                                        └───┘»
   q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
        └───┘└───────────┘                                             »
meas: 4/═══════════════════════════════════════════════════════════════»
                                                                       »
«                                                     »
«   q_0: ────────────────────────────────■─────────■──»
«                                        │         │  »
«   q_1: ────────────────────

In [13]:
import torch

from src.models.quantum_model import create_model
from src.models.hybrid_classifier import HybridClassifier

# ------------------------------------------------------------
# Reconstruct the exact 04B architecture
# ------------------------------------------------------------

quantum_model_04B = create_model(
    num_qubits=4,
    seed=42,
)

model_04B = HybridClassifier(
    quantum_model=quantum_model_04B
)

# ------------------------------------------------------------
# Load trained checkpoint
# ------------------------------------------------------------

checkpoint_path = "../results/models/vqc_04B_scaled_seed42.pt"

state_dict = torch.load(
    checkpoint_path,
    map_location="cpu"
)

model_04B.load_state_dict(state_dict)

model_04B.eval()

print("04B model restored successfully.")
print(model_04B)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


04B model restored successfully.
HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)


In [15]:
import numpy as np

quantum_weights = (
    model_04B.quantum.weight
    .detach()
    .cpu()
    .numpy()
)

classifier_weight = (
    model_04B.classifier.weight
    .detach()
    .cpu()
    .numpy()
    .item()
)

classifier_bias = (
    model_04B.classifier.bias
    .detach()
    .cpu()
    .numpy()
    .item()
)

print("Quantum weights:")
print(quantum_weights)

print("\nNumber of quantum weights:")
print(len(quantum_weights))

print("\nClassifier weight:")
print(classifier_weight)

print("\nClassifier bias:")
print(classifier_bias)

Quantum weights:
[ 1.5234425   1.3247753  -0.30822906  1.6413008  -1.8245659   0.00345224
 -0.04901591  1.5064764   0.8815429  -0.73362815  0.8691962   0.1010682 ]

Number of quantum weights:
12

Classifier weight:
2.050360918045044

Classifier bias:
0.04361313208937645


In [17]:
from qiskit import QuantumCircuit

qc_measure = qc_transpiled.copy()
qc_measure.measure_all()

print(qc_measure.count_ops())

OrderedDict({'cx': 30, 'p': 20, 'ry': 12, 'h': 8, 'measure': 4, 'barrier': 1})


In [18]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    ReadoutError,
)

SHOTS = 1024

P_1Q = 0.001
P_2Q = 0.01
P_READOUT = 0.02

# ============================================================
# Depolarizing noise
# ============================================================

depolarizing_noise = NoiseModel()

error_1q = depolarizing_error(P_1Q, 1)
error_2q = depolarizing_error(P_2Q, 2)

depolarizing_noise.add_all_qubit_quantum_error(
    error_1q,
    ["p", "ry", "h"]
)

depolarizing_noise.add_all_qubit_quantum_error(
    error_2q,
    ["cx"]
)

# ============================================================
# Readout noise
# ============================================================

readout_noise = NoiseModel()

readout_error = ReadoutError([
    [1 - P_READOUT, P_READOUT],
    [P_READOUT, 1 - P_READOUT],
])

readout_noise.add_all_qubit_readout_error(
    readout_error
)

# ============================================================
# Combined
# ============================================================

combined_noise = NoiseModel()

combined_noise.add_all_qubit_quantum_error(
    error_1q,
    ["p", "ry", "h"]
)

combined_noise.add_all_qubit_quantum_error(
    error_2q,
    ["cx"]
)

combined_noise.add_all_qubit_readout_error(
    readout_error
)

# ============================================================
# Simulators
# ============================================================

ideal_simulator = AerSimulator()

depolarizing_simulator = AerSimulator(
    noise_model=depolarizing_noise
)

readout_simulator = AerSimulator(
    noise_model=readout_noise
)

combined_simulator = AerSimulator(
    noise_model=combined_noise
)

print("All simulators created successfully.")

All simulators created successfully.


In [26]:
import numpy as np
import joblib

# ============================================================
# LOAD BINARY PCA DATA
# ============================================================

X_test_pca = np.load(
    "../data/binary/X_test.npy"
)

y_test = np.load(
    "../data/binary/y_test.npy"
)

# ============================================================
# LOAD FROZEN 04B SCALER
# ============================================================

scaler_04B = joblib.load(
    "../results/preprocessing/standard_scaler_04B_seed42.joblib"
)

# ============================================================
# APPLY EXACT 04B STANDARDIZATION
# ============================================================

X_test_scaled_04B = scaler_04B.transform(
    X_test_pca
)

print("X_test_pca shape:", X_test_pca.shape)
print("y_test shape:", y_test.shape)
print("X_test_scaled_04B shape:", X_test_scaled_04B.shape)

print("\nScaled test mean:")
print(X_test_scaled_04B.mean(axis=0))

print("\nScaled test std:")
print(X_test_scaled_04B.std(axis=0))

X_test_pca shape: (2956, 4)
y_test shape: (2956,)
X_test_scaled_04B shape: (2956, 4)

Scaled test mean:
[0.0017654  0.00180763 0.01846003 0.03138886]

Scaled test std:
[1.00099269 1.00601958 0.99919916 0.98443775]


In [27]:
from sklearn.metrics import accuracy_score

saved_predictions = np.load(
    "../results/adversarial/clean_predictions_04B_seed42.npy"
)

print("Saved predictions:", saved_predictions.shape)
print("Labels:", y_test.shape)

print(
    "Saved prediction accuracy:",
    accuracy_score(y_test, saved_predictions)
)

Saved predictions: (2956,)
Labels: (2956,)
Saved prediction accuracy: 0.5852503382949933


In [28]:
# ============================================================
# EXPERIMENT 06A — ONE-SAMPLE SMOKE TEST
# ============================================================

sample = X_test_scaled_04B[0]

print("Sample:")
print(sample)

# Bind the 4 input features + 12 trained quantum parameters
parameter_values = {}

for param, value in zip(
    feature_map_04B.parameters,
    sample
):
    parameter_values[param] = float(value)

for param, value in zip(
    ansatz_04B.parameters,
    quantum_weights
):
    parameter_values[param] = float(value)

print("\nParameters bound:", len(parameter_values))

# Bind parameters to circuit
bound_circuit = qc_measure.assign_parameters(
    parameter_values
)

print("Remaining unbound parameters:")
print(len(bound_circuit.parameters))

Sample:
[ 0.92402527 -1.18636989  0.61522836  0.41028873]

Parameters bound: 16
Remaining unbound parameters:
0


In [42]:
# ============================================================
# Z EXPECTATION
# ============================================================

def calculate_z_expectation_qnn(counts):
    """
    Calculate <ZIII>, matching the QNN observable:

        SparsePauliOp("ZIII")

    In Qiskit's Pauli-string convention, the leftmost
    character corresponds to the highest-index qubit.

    Therefore ZIII means Z on q3.
    """

    total_shots = sum(counts.values())

    zeros = 0
    ones = 0

    for bitstring, count in counts.items():

        # Qiskit displays classical bits as q3 q2 q1 q0.
        # ZIII acts on q3 -> leftmost bit.
        measured_bit = bitstring[0]

        if measured_bit == "0":
            zeros += count
        else:
            ones += count

    return (zeros - ones) / total_shots

In [43]:
for name, simulator in simulators.items():

    job = simulator.run(
        bound_circuit,
        shots=1024,
        seed_simulator=42
    )

    result = job.result()
    counts = result.get_counts()

    z_value = calculate_z_expectation_qnn(counts)

    logit = (
        classifier_weight * z_value
        + classifier_bias
    )

    prediction = int(logit >= 0)

    print("=" * 60)
    print(name.upper())
    print("Z expectation:", z_value)
    print("Logit:", logit)
    print("Prediction:", prediction)

IDEAL
Z expectation: -0.046875
Logit: -0.052497535943984985
Prediction: 0
DEPOLARIZING
Z expectation: -0.025390625
Logit: -0.008446813095360994
Prediction: 0
READOUT
Z expectation: -0.04296875
Logit: -0.04448831360787153
Prediction: 0
COMBINED
Z expectation: -0.025390625
Logit: -0.008446813095360994
Prediction: 0


In [44]:
ideal_validation = evaluate_aer_condition(
    simulator=ideal_simulator,
    condition_name="ideal_corrected",
    X=X_test_scaled_04B,
    y=y_test,
    feature_map=feature_map_04B,
    ansatz=ansatz_04B,
    qc_measure=qc_measure,
    quantum_weights=quantum_weights,
    classifier_weight=classifier_weight,
    classifier_bias=classifier_bias,
    shots=1024,
    batch_size=16,
    seed=42,
)

print()
print("=" * 70)
print("CORRECTED IDEAL AER VALIDATION")
print("=" * 70)

print(f"Accuracy:  {ideal_validation['accuracy']:.6f}")
print(f"Precision: {ideal_validation['precision']:.6f}")
print(f"Recall:    {ideal_validation['recall']:.6f}")
print(f"F1:        {ideal_validation['f1']:.6f}")
print(
    f"Runtime:   "
    f"{ideal_validation['runtime_seconds'] / 60:.2f} min"
)

ideal_corrected: 16/2956
ideal_corrected: 32/2956
ideal_corrected: 48/2956
ideal_corrected: 64/2956
ideal_corrected: 80/2956
ideal_corrected: 96/2956
ideal_corrected: 112/2956
ideal_corrected: 128/2956
ideal_corrected: 144/2956
ideal_corrected: 160/2956
ideal_corrected: 176/2956
ideal_corrected: 192/2956
ideal_corrected: 208/2956
ideal_corrected: 224/2956
ideal_corrected: 240/2956
ideal_corrected: 256/2956
ideal_corrected: 272/2956
ideal_corrected: 288/2956
ideal_corrected: 304/2956
ideal_corrected: 320/2956
ideal_corrected: 336/2956
ideal_corrected: 352/2956
ideal_corrected: 368/2956
ideal_corrected: 384/2956
ideal_corrected: 400/2956
ideal_corrected: 416/2956
ideal_corrected: 432/2956
ideal_corrected: 448/2956
ideal_corrected: 464/2956
ideal_corrected: 480/2956
ideal_corrected: 496/2956
ideal_corrected: 512/2956
ideal_corrected: 528/2956
ideal_corrected: 544/2956
ideal_corrected: 560/2956
ideal_corrected: 576/2956
ideal_corrected: 592/2956
ideal_corrected: 608/2956
ideal_corrected: 6

In [45]:
print("Prediction distribution:")

values, counts = np.unique(
    ideal_validation["predictions"],
    return_counts=True
)

print(
    dict(zip(values, counts))
)

print("\nTrue-label distribution:")

values, counts = np.unique(
    y_test,
    return_counts=True
)

print(
    dict(zip(values, counts))
)

Prediction distribution:
{np.int32(0): np.int64(1332), np.int32(1): np.int64(1624)}

True-label distribution:
{np.int64(0): np.int64(1381), np.int64(1): np.int64(1575)}


In [46]:
from qiskit.quantum_info import Statevector

# Use the SAME first test sample
sample = X_test_scaled_04B[0]

# Build parameter dictionary
parameter_values = {}

for param, value in zip(
    feature_map_04B.parameters,
    sample
):
    parameter_values[param] = float(value)

for param, value in zip(
    ansatz_04B.parameters,
    quantum_weights
):
    parameter_values[param] = float(value)

# IMPORTANT:
# Use the circuit WITHOUT measurement
bound_no_measure = qc_transpiled.assign_parameters(
    parameter_values
)

print("Remaining parameters:", len(bound_no_measure.parameters))

# Exact statevector
state = Statevector.from_instruction(
    bound_no_measure
)

# ------------------------------------------------------------
# Exact Z expectation for q0 and q3
# ------------------------------------------------------------

from qiskit.quantum_info import SparsePauliOp

z_q0 = SparsePauliOp.from_list([
    ("IIIZ", 1.0)
])

z_q3 = SparsePauliOp.from_list([
    ("ZIII", 1.0)
])

expectation_q0 = state.expectation_value(z_q0).real
expectation_q3 = state.expectation_value(z_q3).real

print()
print("=" * 60)
print("EXACT STATEVECTOR EXPECTATIONS")
print("=" * 60)

print("q0 <Z>:", expectation_q0)
print("q3 <Z>:", expectation_q3)

print()
print("Corresponding logits:")

print(
    "q0:",
    classifier_weight * expectation_q0
    + classifier_bias
)

print(
    "q3:",
    classifier_weight * expectation_q3
    + classifier_bias
)

Remaining parameters: 0

EXACT STATEVECTOR EXPECTATIONS
q0 <Z>: 0.051815262522410394
q3 <Z>: -0.06563361520800565

Corresponding logits:
q0: 0.1498531213235708
q3: -0.09095946744312516


In [48]:
import torch

x0 = torch.tensor(
    X_test_scaled_04B[0:1],
    dtype=torch.float32
)

# Recreate the trained QNN
qnn_04B = create_model(
    num_qubits=4,
    seed=42
)

print(qnn_04B)

qnn_04B.weight.data = torch.tensor(
    quantum_weights,
    dtype=torch.float32
)

qnn_04B.eval()

with torch.no_grad():
    qnn_output = qnn_04B(x0)

print()
print("=" * 60)
print("ORIGINAL TRAINED QNN")
print("=" * 60)

print("QNN output:", qnn_output.numpy())

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


TorchConnector()

ORIGINAL TRAINED QNN
QNN output: [[-0.0608724]]


In [50]:
print("=" * 70)
print("MEASUREMENT CIRCUIT")
print("=" * 70)

print(qc_measure)

print()
print("Qubits:", qc_measure.qubits)
print("Clbits:", qc_measure.clbits)

print()
print("Classical registers:")
for creg in qc_measure.cregs:
    print(creg)

print("=" * 70)
print("MEASUREMENT INSTRUCTIONS")
print("=" * 70)

for instruction in qc_measure.data:
    print(instruction)

MEASUREMENT CIRCUIT
        ┌───┐┌───────────┐                                             »
   q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■────■──»
        ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐  │  »
   q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──┼──»
        ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
   q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
        ├───┤├───────────┤                                        └───┘»
   q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
        └───┘└───────────┘                                             »
meas: 4/═══════════════════════════════════════════════════════════════»
                                                                       »
«                                                     »
«   q_0: ────────────────────────────────■─────────■──»
«                                        │         │  »
«   q_1: 

In [109]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_aer_condition_fixed(
    simulator,
    condition_name,
    X,
    y,
    feature_map,
    ansatz,
    quantum_weights,
    classifier_weight,
    classifier_bias,
    shots=1024,
    batch_size=16,
    seed=42
):
    print("=" * 70)
    print(f"RUNNING {condition_name.upper()}")
    print("=" * 70)

    # ------------------------------------------------------------
    # Build the complete measurement circuit
    # ------------------------------------------------------------
    qc = feature_map.compose(ansatz)

    # Measure all qubits
    qc_measure = qc.copy()

    # Remove any existing classical registers if necessary
    qc_measure.measure_all()

    # ------------------------------------------------------------
    # Parameter ordering
    # ------------------------------------------------------------
    feature_params = list(feature_map.parameters)
    ansatz_params = list(ansatz.parameters)

    print("Feature parameters:", len(feature_params))
    print("Ansatz parameters:", len(ansatz_params))
    print("Total parameters:", len(qc.parameters))

    # Convert trained parameters to flat numpy arrays
    quantum_weights = np.asarray(
        quantum_weights,
        dtype=float
    ).reshape(-1)

    classifier_weight = float(
        np.asarray(classifier_weight).reshape(-1)[0]
    )

    classifier_bias = float(
        np.asarray(classifier_bias).reshape(-1)[0]
    )

    print("Quantum weights:", quantum_weights.shape)
    print("Classifier weight:", classifier_weight)
    print("Classifier bias:", classifier_bias)

    assert len(feature_params) == X.shape[1]
    assert len(ansatz_params) == len(quantum_weights)

    predictions = []

    # ------------------------------------------------------------
    # Process samples
    # ------------------------------------------------------------
    for start in range(0, len(X), batch_size):

        end = min(start + batch_size, len(X))

        if start == 0 or end == len(X):
            print(f"{condition_name}: {end}/{len(X)}")

        circuits = []

        for x in X[start:end]:

            # ----------------------------------------------------
            # Create complete binding
            # ----------------------------------------------------
            bind_dict = {}

            # Input features
            for param, value in zip(feature_params, x):
                bind_dict[param] = float(value)

            # Trained quantum parameters
            for param, value in zip(
                ansatz_params,
                quantum_weights
            ):
                bind_dict[param] = float(value)

            # Bind everything before sending to Aer
            bound_circuit = qc_measure.assign_parameters(
                bind_dict,
                inplace=False
            )

            circuits.append(bound_circuit)

        # --------------------------------------------------------
        # Run completely parameter-free circuits
        # --------------------------------------------------------
        job = simulator.run(
            circuits,
            shots=shots,
            seed_simulator=seed + start
        )

        result = job.result()

        # --------------------------------------------------------
        # Extract predictions
        # --------------------------------------------------------
        for i in range(len(circuits)):

            counts = result.get_counts(i)

            total = sum(counts.values())

            # Qiskit bit strings are displayed MSB -> LSB.
            # We measure q0 -> classical bit 0.
            #
            # For the original model, use q0 as the observable.
            expectation = 0.0

            for bitstring, count in counts.items():

                # Remove spaces if present
                bitstring = bitstring.replace(" ", "")

                # q0 corresponds to the RIGHTMOST bit
                q0 = int(bitstring[-1])

                z_value = 1.0 if q0 == 0 else -1.0

                expectation += (
                    z_value * count / total
                )

            # ----------------------------------------------------
            # Same classical readout as trained model
            # ----------------------------------------------------
            logit = (
                classifier_weight * expectation
                + classifier_bias
            )

            prediction = 1 if logit >= 0 else 0

            predictions.append(prediction)

    predictions = np.asarray(predictions)

    # ------------------------------------------------------------
    # Metrics
    # ------------------------------------------------------------
    accuracy = accuracy_score(y, predictions)

    precision = precision_score(
        y,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y,
        predictions,
        zero_division=0
    )

    print()
    print(f"{condition_name.upper()} COMPLETE")
    print(f"Accuracy:  {accuracy:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall:    {recall:.6f}")
    print(f"F1:        {f1:.6f}")

    return {
        "condition": condition_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "predictions": predictions
    }

In [111]:
# ============================================================
# COMPLETE 04B QUANTUM MODEL VERIFICATION
# ============================================================

import numpy as np
import torch

from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit.primitives import StatevectorEstimator


# ============================================================
# 1. SELECT THE TRAINED MODEL
# ============================================================

# Your notebook already contains:
# model_04B
#
# This is the HybridClassifier containing:
#   model_04B.quantum
#   model_04B.classifier

model = model_04B

torch_qnn = model.quantum

print("=" * 70)
print("MODEL")
print("=" * 70)
print(model)

print("\nQuantum component:")
print(torch_qnn)


# ============================================================
# 2. EXTRACT TRAINED PARAMETERS
# ============================================================

quantum_weights = (
    torch_qnn.weight
    .detach()
    .cpu()
    .numpy()
    .astype(float)
    .flatten()
)

classifier_weight = (
    model.classifier.weight
    .detach()
    .cpu()
    .numpy()
    .flatten()[0]
)

classifier_bias = (
    model.classifier.bias
    .detach()
    .cpu()
    .numpy()
    .flatten()[0]
)

print("\n" + "=" * 70)
print("TRAINED PARAMETERS")
print("=" * 70)

print("Quantum weights:")
print(quantum_weights)

print("\nNumber of quantum weights:", len(quantum_weights))

print("\nClassifier weight:")
print(classifier_weight)

print("\nClassifier bias:")
print(classifier_bias)


# ============================================================
# 3. GET THE UNDERLYING ESTIMATOR QNN
# ============================================================

estimator_qnn = torch_qnn.neural_network

print("\n" + "=" * 70)
print("UNDERLYING ESTIMATOR QNN")
print("=" * 70)

print(type(estimator_qnn))

print("\nObservable:")
print(estimator_qnn.observables)

print("\nInput parameters:")
print(estimator_qnn.input_params)

print("\nWeight parameters:")
print(estimator_qnn.weight_params)

print("\nNumber of weights:")
print(estimator_qnn.num_weights)


# ============================================================
# 4. USE ONE TEST SAMPLE
# ============================================================

# This uses the same scaled PCA data from Experiment 04B.
#
# If X_test_scaled_04B exists, use it.
# Otherwise this will tell you what is missing.

x_single = np.asarray(
    X_test_scaled_04B[0],
    dtype=float
)

x_single = x_single.reshape(1, 4)

print("\n" + "=" * 70)
print("TEST INPUT")
print("=" * 70)

print("Shape:", x_single.shape)
print("Input:")
print(x_single)


# ============================================================
# 5. TORCHCONNECTOR OUTPUT
# ============================================================

x_tensor = torch.tensor(
    x_single,
    dtype=torch.float32
)

with torch.no_grad():
    torch_output = torch_qnn(x_tensor)

torch_output_value = float(
    torch_output.detach().cpu().numpy().flatten()[0]
)

print("\n" + "=" * 70)
print("TORCHCONNECTOR")
print("=" * 70)

print("Quantum output:")
print(torch_output_value)


# ============================================================
# 6. DIRECT ESTIMATOR QNN OUTPUT
# ============================================================

estimator_output = estimator_qnn.forward(
    x_single,
    quantum_weights
)

estimator_output_value = float(
    np.asarray(estimator_output).flatten()[0]
)

print("\n" + "=" * 70)
print("DIRECT ESTIMATOR QNN")
print("=" * 70)

print("Output:")
print(estimator_output_value)


# ============================================================
# 7. RECREATE EXACT CIRCUIT
# ============================================================

feature_map = ZZFeatureMap(
    feature_dimension=4,
    reps=1
)

ansatz = RealAmplitudes(
    num_qubits=4,
    reps=2
)

exact_circuit = QuantumCircuit(4)

exact_circuit.compose(
    feature_map,
    inplace=True
)

exact_circuit.compose(
    ansatz,
    inplace=True
)

print("\n" + "=" * 70)
print("RECREATED CIRCUIT")
print("=" * 70)

print(exact_circuit)

print("\nOriginal parameters:")
print(exact_circuit.parameters)


# ============================================================
# 8. IMPORTANT:
#    BIND PARAMETERS USING THE PARAMETERS FROM THIS CIRCUIT
# ============================================================

# DO NOT use estimator_qnn.input_params directly here.
#
# The previous CircuitError happened because parameters such as
# x[0] and theta[0] came from a DIFFERENT circuit object.
#
# We therefore use the parameters belonging to exact_circuit.

circuit_parameters = list(exact_circuit.parameters)

print("\nNumber of circuit parameters:")
print(len(circuit_parameters))


# ============================================================
# 9. IDENTIFY INPUT AND WEIGHT PARAMETERS
# ============================================================

input_parameters = sorted(
    [
        p for p in circuit_parameters
        if str(p).startswith("x")
    ],
    key=lambda p: int(str(p).split("[")[1].split("]")[0])
)

weight_parameters = sorted(
    [
        p for p in circuit_parameters
        if str(p).startswith("θ")
    ],
    key=lambda p: int(str(p).split("[")[1].split("]")[0])
)

print("\nInput parameters:")
print(input_parameters)

print("\nWeight parameters:")
print(weight_parameters)

print(
    "\nNumber of input parameters:",
    len(input_parameters)
)

print(
    "Number of weight parameters:",
    len(weight_parameters)
)


# ============================================================
# 10. CREATE BINDING DICTIONARY
# ============================================================

bind_dict = {}

# Input features
for param, value in zip(
    input_parameters,
    x_single.flatten()
):
    bind_dict[param] = float(value)

# Quantum trainable parameters
for param, value in zip(
    weight_parameters,
    quantum_weights
):
    bind_dict[param] = float(value)


# ============================================================
# 11. BIND EVERYTHING
# ============================================================

bound_circuit = exact_circuit.assign_parameters(
    bind_dict
)

print("\n" + "=" * 70)
print("BOUND CIRCUIT")
print("=" * 70)

print(bound_circuit)

print("\nRemaining parameters:")
print(len(bound_circuit.parameters))

if len(bound_circuit.parameters) != 0:
    raise RuntimeError(
        "Circuit still contains unbound parameters."
    )


# ============================================================
# 12. EXACT STATEVECTOR
# ============================================================

statevector = Statevector.from_instruction(
    bound_circuit
)

print("\n" + "=" * 70)
print("STATEVECTOR")
print("=" * 70)

print("Number of amplitudes:")
print(len(statevector))


# ============================================================
# 13. CORRECT OBSERVABLE
# ============================================================

# Your EstimatorQNN reports:
#
#     SparsePauliOp(['ZIII'])
#
# In Qiskit's Pauli-string convention, the LEFTMOST
# character corresponds to the highest-index qubit.
#
# Therefore:
#
#     ZIII -> Z expectation of q3
#
# This is why we measure q3.

observable = SparsePauliOp(
    ["ZIII"],
    coeffs=[1.0]
)

print("\n" + "=" * 70)
print("OBSERVABLE")
print("=" * 70)

print(observable)


# ============================================================
# 14. EXACT ZIII EXPECTATION
# ============================================================

exact_expectation = float(
    np.real(
        statevector.expectation_value(observable)
    )
)

print("\n" + "=" * 70)
print("EXACT STATEVECTOR ZIII")
print("=" * 70)

print(
    f"ZIII expectation = {exact_expectation:.12f}"
)


# ============================================================
# 15. CONVERT QUANTUM OUTPUT TO CLASSIFIER LOGIT
# ============================================================

exact_logit = (
    classifier_weight * exact_expectation
    + classifier_bias
)

exact_prediction = int(
    exact_logit >= 0
)

print("\n" + "=" * 70)
print("EXACT STATEVECTOR → CLASSIFIER")
print("=" * 70)

print(
    f"Z expectation: {exact_expectation:.12f}"
)

print(
    f"Classifier weight: {classifier_weight:.12f}"
)

print(
    f"Classifier bias: {classifier_bias:.12f}"
)

print(
    f"Logit: {exact_logit:.12f}"
)

print(
    f"Prediction: {exact_prediction}"
)


# ============================================================
# 16. CREATE MEASUREMENT CIRCUIT
# ============================================================

qc_measure = bound_circuit.copy()

qc_measure.measure_all()

print("\n" + "=" * 70)
print("MEASUREMENT CIRCUIT")
print("=" * 70)

print(
    qc_measure.count_ops()
)


# ============================================================
# 17. DECOMPOSE FOR AER
# ============================================================

qc_aer = qc_measure.decompose(
    reps=10
)

print("\nAfter decomposition:")
print(qc_aer.count_ops())

print("\nRemaining parameters:")
print(len(qc_aer.parameters))

if len(qc_aer.parameters) != 0:
    raise RuntimeError(
        "Aer circuit still contains parameters."
    )


# ============================================================
# 18. RUN AER
# ============================================================

simulator = AerSimulator()

result = simulator.run(
    qc_aer,
    shots=1024,
    seed_simulator=42
).result()

counts = result.get_counts()


# ============================================================
# 19. PRINT COUNTS
# ============================================================

print("\n" + "=" * 70)
print("AER COUNTS")
print("=" * 70)

print(counts)


# ============================================================
# 20. CALCULATE Z EXPECTATION FOR q3
# ============================================================

# Qiskit measurement strings are displayed as:
#
# q3 q2 q1 q0
#
# Since we need ZIII = Z on q3:
#
# q3 = 0 -> +1
# q3 = 1 -> -1

shots = sum(counts.values())

z_sum = 0.0

for bitstring, count in counts.items():

    bitstring = bitstring.replace(" ", "")

    q3_bit = int(bitstring[0])

    eigenvalue = (
        1.0 if q3_bit == 0
        else -1.0
    )

    z_sum += eigenvalue * count


aer_expectation = z_sum / shots


# ============================================================
# 21. AER → CLASSIFIER
# ============================================================

aer_logit = (
    classifier_weight * aer_expectation
    + classifier_bias
)

aer_prediction = int(
    aer_logit >= 0
)

print("\n" + "=" * 70)
print("AER ZIII MEASUREMENT")
print("=" * 70)

print("Shots:", shots)

print(
    f"ZIII / q3 expectation: "
    f"{aer_expectation:.12f}"
)

print(
    f"Logit: {aer_logit:.12f}"
)

print(
    f"Prediction: {aer_prediction}"
)


# ============================================================
# 22. FINAL COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("FINAL COMPARISON")
print("=" * 70)

print(
    f"{'Method':<30} {'Quantum output':>18}"
)

print("-" * 52)

print(
    f"{'TorchConnector':<30} "
    f"{torch_output_value:>18.12f}"
)

print(
    f"{'EstimatorQNN':<30} "
    f"{estimator_output_value:>18.12f}"
)

print(
    f"{'Exact Statevector ZIII':<30} "
    f"{exact_expectation:>18.12f}"
)

print(
    f"{'Aer ZIII (1024 shots)':<30} "
    f"{aer_expectation:>18.12f}"
)


print("\n" + "=" * 70)
print("FINAL PREDICTIONS")
print("=" * 70)

print(
    f"TorchConnector quantum output: "
    f"{torch_output_value:.12f}"
)

print(
    f"Exact classifier prediction: "
    f"{exact_prediction}"
)

print(
    f"Aer classifier prediction: "
    f"{aer_prediction}"
)


# ============================================================
# 23. DIFFERENCES
# ============================================================

print("\n" + "=" * 70)
print("DIFFERENCES")
print("=" * 70)

print(
    "TorchConnector vs EstimatorQNN:",
    abs(
        torch_output_value
        - estimator_output_value
    )
)

print(
    "EstimatorQNN vs exact statevector:",
    abs(
        estimator_output_value
        - exact_expectation
    )
)

print(
    "Exact statevector vs Aer:",
    abs(
        exact_expectation
        - aer_expectation
    )
)

print("=" * 70)
print("VERIFICATION COMPLETE")
print("=" * 70)

MODEL
HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)

Quantum component:
TorchConnector()

TRAINED PARAMETERS
Quantum weights:
[ 1.52344251  1.32477534 -0.30822906  1.6413008  -1.82456589  0.00345224
 -0.04901591  1.5064764   0.88154292 -0.73362815  0.86919618  0.1010682 ]

Number of quantum weights: 12

Classifier weight:
2.050361

Classifier bias:
0.043613132

UNDERLYING ESTIMATOR QNN
<class 'qiskit_machine_learning.neural_networks.estimator_qnn.EstimatorQNN'>

Observable:
(SparsePauliOp(['ZIII'],
              coeffs=[1.+0.j]),)

Input parameters:
[ParameterVectorElement(x[0]), ParameterVectorElement(x[1]), ParameterVectorElement(x[2]), ParameterVectorElement(x[3])]

Weight parameters:
[ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2]), ParameterVectorElement(θ[3]), ParameterVectorElement(θ[4]), ParameterVectorElement(θ[5]), ParameterVectorElement(θ[6]), ParameterVectorElement(

C:\Users\jneelam\AppData\Local\Temp\ipykernel_14116\4042247932.py:186: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(
C:\Users\jneelam\AppData\Local\Temp\ipykernel_14116\4042247932.py:191: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(


In [53]:
from qiskit.quantum_info import Statevector, SparsePauliOp
import numpy as np

# Use the same input sample
x = X_test_scaled_04B[0]

# Build circuit
qc_diag = feature_map_04B.compose(ansatz_04B)

# Bind input + trained weights
bind_dict = {}

for p, v in zip(feature_map_04B.parameters, x):
    bind_dict[p] = float(v)

for p, v in zip(ansatz_04B.parameters, quantum_weights):
    bind_dict[p] = float(v)

qc_bound = qc_diag.assign_parameters(
    bind_dict,
    inplace=False
)

# Statevector
state = Statevector.from_instruction(qc_bound)

print("Statevector calculated.")
print()

# Test every single-qubit Z observable
for q in range(4):

    pauli = ["I"] * 4
    pauli[3 - q] = "Z"

    observable = SparsePauliOp.from_list([
        ("".join(pauli), 1.0)
    ])

    expectation = np.real(
        state.expectation_value(observable)
    )

    logit = (
        classifier_weight * expectation
        + classifier_bias
    )

    prediction = int(logit >= 0)

    print(
        f"q{q} <Z> = {expectation:.10f} | "
        f"logit = {logit:.10f} | "
        f"prediction = {prediction}"
    )

Statevector calculated.

q0 <Z> = 0.0518152625 | logit = 0.1498531213 | prediction = 1
q1 <Z> = 0.0377889356 | logit = 0.1210940887 | prediction = 1
q2 <Z> = 0.0880390258 | logit = 0.2241249098 | prediction = 1
q3 <Z> = -0.0656336152 | logit = -0.0909594674 | prediction = 0


In [54]:
output_original = model_04B.quantum(
    torch.tensor(
        X_test_scaled_04B[:1],
        dtype=torch.float32
    )
)

print("Original TorchConnector output:")
print(output_original.detach().cpu().numpy())

Original TorchConnector output:
[[-0.0608724]]


In [57]:
# ============================================================
# INSPECT ESTIMATORQNN INTERNALS
# ============================================================

import inspect

connector = model_04B.quantum
qnn = connector.neural_network

print("=" * 70)
print("QNN TYPE")
print("=" * 70)
print(type(qnn))

print("\n" + "=" * 70)
print("QNN __dict__")
print("=" * 70)

for key, value in qnn.__dict__.items():
    print(f"\n{key}:")
    print("  type:", type(value))
    print("  value:", value)

print("\n" + "=" * 70)
print("QNN ATTRIBUTES CONTAINING 'OBS', 'EST', OR 'PRIM'")
print("=" * 70)

for name in dir(qnn):
    name_lower = name.lower()

    if (
        "obs" in name_lower
        or "est" in name_lower
        or "prim" in name_lower
    ):
        try:
            value = getattr(qnn, name)
            print(f"{name}: {type(value)}")
        except Exception as e:
            print(f"{name}: <error: {e}>")

print("\n" + "=" * 70)
print("ESTIMATORQNN CONSTRUCTOR")
print("=" * 70)

print(inspect.signature(type(qnn).__init__))

QNN TYPE
<class 'qiskit_machine_learning.neural_networks.estimator_qnn.EstimatorQNN'>

QNN __dict__

estimator:
  type: <class 'qiskit.primitives.statevector_estimator.StatevectorEstimator'>
  value: <qiskit.primitives.statevector_estimator.StatevectorEstimator object at 0x000001E00DE170E0>

num_virtual_qubits:
  type: <class 'int'>
  value: 4

_org_circuit:
  type: <class 'qiskit.circuit.quantumcircuit.QuantumCircuit'>
  value:      ┌────────────────────────────────────┐»
q_0: ┤0                                   ├»
     │                                    │»
q_1: ┤1                                   ├»
     │  ZZFeatureMap(x[0],x[1],x[2],x[3]) │»
q_2: ┤2                                   ├»
     │                                    │»
q_3: ┤3                                   ├»
     └────────────────────────────────────┘»
«     ┌────────────────────────────────────────────────────────────────────────────────┐
«q_0: ┤0                                                                 

In [58]:
qnn = model_04B.quantum.neural_network

print("Observable:")
print(qnn._observables)

print("\nObservable Pauli terms:")
print(qnn._observables[0].to_list())

Observable:
(SparsePauliOp(['ZIII'],
              coeffs=[1.+0.j]),)

Observable Pauli terms:
[('ZIII', (1+0j))]


In [59]:
import torch
import numpy as np

x_test = torch.tensor(
    X_test_scaled_04B[:1],
    dtype=torch.float32
)

with torch.no_grad():
    original_output = model_04B.quantum(x_test)

print("Original TorchConnector:")
print(original_output.detach().cpu().numpy())

Original TorchConnector:
[[-0.0608724]]


In [60]:
qnn = model_04B.quantum.neural_network

x_numpy = X_test_scaled_04B[:1]

weights_numpy = (
    model_04B.quantum.weight
    .detach()
    .cpu()
    .numpy()
)

print("Input shape:", x_numpy.shape)
print("Weights shape:", weights_numpy.shape)

qnn_output = qnn.forward(
    x_numpy,
    weights_numpy
)

print("\nDirect EstimatorQNN output:")
print(qnn_output)

Input shape: (1, 4)
Weights shape: (12,)

Direct EstimatorQNN output:
[[-0.06087241]]


In [63]:
SparsePauliOp(["ZIII"])

SparsePauliOp(['ZIII'],
              coeffs=[1.+0.j])

In [67]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# ============================================================
# EXACT ONE-SAMPLE AER VALIDATION
# ============================================================

x = X_test_scaled_04B[0]

# ------------------------------------------------------------
# Extract trained parameters
# ------------------------------------------------------------

quantum_weights = (
    model_04B.quantum.weight
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

classifier_weight = float(
    model_04B.classifier.weight
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)[0]
)

classifier_bias = float(
    model_04B.classifier.bias
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)[0]
)

print("Quantum weights:", quantum_weights.shape)
print("Classifier weight:", classifier_weight)
print("Classifier bias:", classifier_bias)

# ------------------------------------------------------------
# Build original circuit
# ------------------------------------------------------------

qc = feature_map_04B.compose(ansatz_04B)

# ------------------------------------------------------------
# Bind parameters
# ------------------------------------------------------------

bind_dict = {}

for p, value in zip(feature_map_04B.parameters, x):
    bind_dict[p] = float(value)

for p, value in zip(ansatz_04B.parameters, quantum_weights):
    bind_dict[p] = float(value)

qc_bound = qc.assign_parameters(
    bind_dict,
    inplace=False
)

# ------------------------------------------------------------
# Add measurement
# ------------------------------------------------------------

qc_measure = qc_bound.copy()

qc_measure.measure_all()

print("\nParameters remaining:")
print(len(qc_measure.parameters))

# ------------------------------------------------------------
# Run Aer
# ------------------------------------------------------------
# ============================================================
# DECOMPOSE CIRCUIT FOR AER
# ============================================================

from qiskit import transpile

qc_aer = transpile(
    qc_measure,
    simulator,
    optimization_level=0
)

print("Transpiled circuit:")
print(qc_aer)

result = simulator.run(
    qc_aer,
    shots=1024,
    seed_simulator=42
).result()

counts = result.get_counts()

print("\nCounts:")
print(counts)

# ------------------------------------------------------------
# IMPORTANT:
# ZIII = Z on q3
#
# Qiskit count strings are q3 q2 q1 q0
# Therefore q3 is the LEFTMOST bit.
# ------------------------------------------------------------

p_q3_0 = 0
p_q3_1 = 0

total = sum(counts.values())

for bitstring, count in counts.items():

    bitstring = bitstring.replace(" ", "")

    q3 = int(bitstring[0])

    if q3 == 0:
        p_q3_0 += count
    else:
        p_q3_1 += count

p_q3_0 /= total
p_q3_1 /= total

z_expectation = p_q3_0 - p_q3_1

print("\nP(q3=0):", p_q3_0)
print("P(q3=1):", p_q3_1)

print("\nAer ZIII expectation:")
print(z_expectation)

# ------------------------------------------------------------
# Classifier
# ------------------------------------------------------------

aer_logit = (
    classifier_weight * z_expectation
    + classifier_bias
)

aer_prediction = int(aer_logit >= 0)

print("\nAer logit:")
print(aer_logit)

print("Aer prediction:")
print(aer_prediction)

# ------------------------------------------------------------
# Original QNN
# ------------------------------------------------------------

import torch

with torch.no_grad():

    original_qnn_output = (
        model_04B.quantum(
            torch.tensor(
                X_test_scaled_04B[:1],
                dtype=torch.float32
            )
        )
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)[0]
    )

print("\nOriginal QNN output:")
print(original_qnn_output)

print("\nDifference:")
print(
    abs(
        original_qnn_output - aer_logit
    )
)

Quantum weights: (12,)
Classifier weight: 2.050360918045044
Classifier bias: 0.04361313208937645

Parameters remaining:
0
Transpiled circuit:
        ┌───┐┌───────────┐                                                    »
   q_0: ┤ H ├┤ P(1.8481) ├───■─────────────────■────■─────────────────■───────»
        ├───┤├───────────┴┐┌─┴─┐┌───────────┐┌─┴─┐  │                 │       »
   q_1: ┤ H ├┤ P(-2.3727) ├┤ X ├┤ P(19.195) ├┤ X ├──┼─────────────────┼────■──»
        ├───┤├───────────┬┘└───┘└───────────┘└───┘┌─┴─┐┌───────────┐┌─┴─┐┌─┴─┐»
   q_2: ┤ H ├┤ P(1.2305) ├────────────────────────┤ X ├┤ P(11.205) ├┤ X ├┤ X ├»
        ├───┤├───────────┴┐                       └───┘└───────────┘└───┘└───┘»
   q_3: ┤ H ├┤ P(0.82058) ├───────────────────────────────────────────────────»
        └───┘└────────────┘                                                   »
meas: 4/══════════════════════════════════════════════════════════════════════»
                                                          

In [69]:
def z_expectation_from_counts(counts, qubit=3):
    total_shots = sum(counts.values())

    expectation = 0.0

    for bitstring, count in counts.items():

        # Qiskit displays bitstrings as q3 q2 q1 q0
        bit = bitstring[-1 - qubit]

        if bit == "0":
            expectation += count / total_shots
        else:
            expectation -= count / total_shots

    return expectation

z_exp_q3 = z_expectation_from_counts(
    counts,
    qubit=3
)

print("Z expectation q3:", z_exp_q3)

print("=" * 60)
print("AER VS TRAINED QNN")
print("=" * 60)

print("Aer q3 expectation:", z_exp_q3)

print("Original TorchConnector output:",
      float(torch_qnn_output.squeeze()))

print("Difference:",
      z_exp_q3 - float(torch_qnn_output.squeeze()))

Z expectation q3: -0.025390625
AER VS TRAINED QNN
Aer q3 expectation: -0.025390625


NameError: name 'torch_qnn_output' is not defined

In [70]:
import numpy as np
from qiskit import transpile

# ============================================================
# EXACT QNN CIRCUIT -> AER DIAGNOSTIC
# ============================================================

# Extract the exact circuit used by EstimatorQNN
exact_circuit = qnn._circuit.copy()

print("Circuit parameters:")
print(exact_circuit.parameters)

print("\nNumber of parameters:")
print(len(exact_circuit.parameters))

print("\nQNN input parameters:")
print(qnn.input_params)

print("\nQNN weight parameters:")
print(qnn.weight_params)

Circuit parameters:
ParameterView([ParameterVectorElement(inputs[0]), ParameterVectorElement(inputs[1]), ParameterVectorElement(inputs[2]), ParameterVectorElement(inputs[3]), ParameterVectorElement(weights[0]), ParameterVectorElement(weights[1]), ParameterVectorElement(weights[2]), ParameterVectorElement(weights[3]), ParameterVectorElement(weights[4]), ParameterVectorElement(weights[5]), ParameterVectorElement(weights[6]), ParameterVectorElement(weights[7]), ParameterVectorElement(weights[8]), ParameterVectorElement(weights[9]), ParameterVectorElement(weights[10]), ParameterVectorElement(weights[11])])

Number of parameters:
16

QNN input parameters:
[ParameterVectorElement(x[0]), ParameterVectorElement(x[1]), ParameterVectorElement(x[2]), ParameterVectorElement(x[3])]

QNN weight parameters:
[ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2]), ParameterVectorElement(θ[3]), ParameterVectorElement(θ[4]), ParameterVectorElement(θ[5]), ParameterVector

In [75]:
import torch

print("Searching for TorchConnector / model objects...")
print("=" * 60)

for name, obj in list(globals().items()):
    try:
        if isinstance(obj, torch.nn.Module):
            print(f"{name}: {type(obj)}")
    except Exception:
        pass

Searching for TorchConnector / model objects...
quantum_model_04B: <class 'qiskit_machine_learning.connectors.torch_connector.TorchConnector'>
model_04B: <class 'src.models.hybrid_classifier.HybridClassifier'>
qnn_04B: <class 'qiskit_machine_learning.connectors.torch_connector.TorchConnector'>
connector: <class 'qiskit_machine_learning.connectors.torch_connector.TorchConnector'>


In [79]:
import torch
import numpy as np

# Use the actual trained hybrid model
torch_qnn = model_04B.quantum

print("=" * 70)
print("MODEL 04B")
print("=" * 70)
print(model_04B)

print("\n" + "=" * 70)
print("TRAINED QUANTUM PARAMETERS")
print("=" * 70)

for name, value in torch_qnn.named_parameters():
    print(f"{name}:")
    print(value.detach().cpu().numpy())

MODEL 04B
HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)

TRAINED QUANTUM PARAMETERS
weight:
[ 1.5234425   1.3247753  -0.30822906  1.6413008  -1.8245659   0.00345224
 -0.04901591  1.5064764   0.8815429  -0.73362815  0.8691962   0.1010682 ]


In [80]:
print("=" * 70)
print("ALL TRAINED MODEL PARAMETERS")
print("=" * 70)

for name, value in model_04B.named_parameters():
    print(f"{name}:")
    print(value.detach().cpu().numpy())

ALL TRAINED MODEL PARAMETERS
quantum.weight:
[ 1.5234425   1.3247753  -0.30822906  1.6413008  -1.8245659   0.00345224
 -0.04901591  1.5064764   0.8815429  -0.73362815  0.8691962   0.1010682 ]
classifier.weight:
[[2.050361]]
classifier.bias:
[0.04361313]


In [81]:
x_single = np.asarray(X_test_scaled_04B[0], dtype=np.float64)

x_tensor = torch.tensor(
    x_single.reshape(1, -1),
    dtype=torch.float32
)

model_04B.eval()

with torch.no_grad():
    q_output = model_04B.quantum(x_tensor)
    final_output = model_04B(x_tensor)

print("=" * 70)
print("MODEL 04B VERIFICATION")
print("=" * 70)

print("Input:")
print(x_single)

print("\nTorchConnector output:")
print(q_output.detach().cpu().numpy())

print("\nFull HybridClassifier output:")
print(final_output.detach().cpu().numpy())

MODEL 04B VERIFICATION
Input:
[ 0.92402527 -1.18636989  0.61522836  0.41028873]

TorchConnector output:
[[-0.0608724]]

Full HybridClassifier output:
[[-0.08119726]]


In [83]:
print("=" * 70)
print("UNDERLYING ESTIMATOR QNN")
print("=" * 70)

print(type(torch_qnn.neural_network))
print(torch_qnn.neural_network)

estimator_qnn_04B = torch_qnn.neural_network

print("\nObservable:")
print(estimator_qnn_04B.observables)

print("\nInput parameters:")
print(estimator_qnn_04B.input_params)

print("\nWeight parameters:")
print(estimator_qnn_04B.weight_params)

print("\nNumber of weights:")
print(estimator_qnn_04B.num_weights)

UNDERLYING ESTIMATOR QNN
<class 'qiskit_machine_learning.neural_networks.estimator_qnn.EstimatorQNN'>

Observable:
(SparsePauliOp(['ZIII'],
              coeffs=[1.+0.j]),)

Input parameters:
[ParameterVectorElement(x[0]), ParameterVectorElement(x[1]), ParameterVectorElement(x[2]), ParameterVectorElement(x[3])]

Weight parameters:
[ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2]), ParameterVectorElement(θ[3]), ParameterVectorElement(θ[4]), ParameterVectorElement(θ[5]), ParameterVectorElement(θ[6]), ParameterVectorElement(θ[7]), ParameterVectorElement(θ[8]), ParameterVectorElement(θ[9]), ParameterVectorElement(θ[10]), ParameterVectorElement(θ[11])]

Number of weights:
12


In [84]:
# ============================================================
# EXTRACT EXACT TRAINED QNN CIRCUIT
# ============================================================

estimator_qnn_04B = model_04B.quantum.neural_network

exact_circuit = estimator_qnn_04B._circuit
observable = estimator_qnn_04B._observables[0]

print("=" * 70)
print("EXACT QNN CIRCUIT")
print("=" * 70)

print(exact_circuit)

print("\n" + "=" * 70)
print("CIRCUIT PARAMETERS")
print("=" * 70)

print("Input parameters:")
print(exact_circuit.parameters)

print("\nNumber of parameters:")
print(len(exact_circuit.parameters))

print("\nObservable:")
print(observable)

EXACT QNN CIRCUIT
     ┌────────────────────────────────────────────────────────┐»
q_0: ┤0                                                       ├»
     │                                                        │»
q_1: ┤1                                                       ├»
     │  ZZFeatureMap(inputs[0],inputs[1],inputs[2],inputs[3]) │»
q_2: ┤2                                                       ├»
     │                                                        │»
q_3: ┤3                                                       ├»
     └────────────────────────────────────────────────────────┘»
«     ┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
«q_0: ┤0                                                                                                                                                       ├
«     │                                                                    

In [85]:
# ============================================================
# EXACT PARAMETER BINDING
# ============================================================

x_single = np.asarray(
    X_test_scaled_04B[0],
    dtype=float
)

theta = (
    model_04B.quantum.weight
    .detach()
    .cpu()
    .numpy()
    .flatten()
)

print("Input x:")
print(x_single)

print("\nTrained theta:")
print(theta)

print("\nNumber of theta values:")
print(len(theta))

Input x:
[ 0.92402527 -1.18636989  0.61522836  0.41028873]

Trained theta:
[ 1.5234425   1.3247753  -0.30822906  1.6413008  -1.8245659   0.00345224
 -0.04901591  1.5064764   0.8815429  -0.73362815  0.8691962   0.1010682 ]

Number of theta values:
12


In [87]:
# ============================================================
# CORRECT BINDING OF THE INTERNAL QNN CIRCUIT
# ============================================================

import numpy as np

x_single = np.asarray(
    X_test_scaled_04B[0],
    dtype=float
)

theta = (
    model_04B.quantum.weight
    .detach()
    .cpu()
    .numpy()
    .flatten()
)

print("Input values:")
print(x_single)

print("\nQuantum weights:")
print(theta)

print("\nInternal circuit parameters:")
for i, param in enumerate(exact_circuit.parameters):
    print(i, repr(param), str(param))

print("\nTotal circuit parameters:", len(exact_circuit.parameters))

Input values:
[ 0.92402527 -1.18636989  0.61522836  0.41028873]

Quantum weights:
[ 1.5234425   1.3247753  -0.30822906  1.6413008  -1.8245659   0.00345224
 -0.04901591  1.5064764   0.8815429  -0.73362815  0.8691962   0.1010682 ]

Internal circuit parameters:
0 ParameterVectorElement(inputs[0]) inputs[0]
1 ParameterVectorElement(inputs[1]) inputs[1]
2 ParameterVectorElement(inputs[2]) inputs[2]
3 ParameterVectorElement(inputs[3]) inputs[3]
4 ParameterVectorElement(weights[0]) weights[0]
5 ParameterVectorElement(weights[1]) weights[1]
6 ParameterVectorElement(weights[2]) weights[2]
7 ParameterVectorElement(weights[3]) weights[3]
8 ParameterVectorElement(weights[4]) weights[4]
9 ParameterVectorElement(weights[5]) weights[5]
10 ParameterVectorElement(weights[6]) weights[6]
11 ParameterVectorElement(weights[7]) weights[7]
12 ParameterVectorElement(weights[8]) weights[8]
13 ParameterVectorElement(weights[9]) weights[9]
14 ParameterVectorElement(weights[10]) weights[10]
15 ParameterVectorElem

In [89]:
# ============================================================
# BUILD BINDING USING INTERNAL CIRCUIT PARAMETERS
# ============================================================

bind_dict_internal = {}

for param in exact_circuit.parameters:
    name = str(param)

    if name.startswith("inputs["):
        index = int(name.split("[")[1].split("]")[0])
        bind_dict_internal[param] = float(x_single[index])

    elif name.startswith("weights["):
        index = int(name.split("[")[1].split("]")[0])
        bind_dict_internal[param] = float(theta[index])

    else:
        raise ValueError(f"Unexpected parameter in circuit: {param}")

print("=" * 70)
print("INTERNAL CIRCUIT BINDINGS")
print("=" * 70)

for param, value in bind_dict_internal.items():
    print(f"{param} = {value}")

bound_circuit = exact_circuit.assign_parameters(
    bind_dict_internal
)

print("\nRemaining parameters:", len(bound_circuit.parameters))

INTERNAL CIRCUIT BINDINGS
inputs[0] = 0.924025274230905
inputs[1] = -1.1863698900134665
inputs[2] = 0.6152283562486526
inputs[3] = 0.410288734555093
weights[0] = 1.5234425067901611
weights[1] = 1.3247753381729126
weights[2] = -0.3082290589809418
weights[3] = 1.6413007974624634
weights[4] = -1.8245658874511719
weights[5] = 0.003452243749052286
weights[6] = -0.04901590943336487
weights[7] = 1.5064764022827148
weights[8] = 0.8815429210662842
weights[9] = -0.7336281538009644
weights[10] = 0.8691961765289307
weights[11] = 0.10106819868087769

Remaining parameters: 0


In [96]:
from qiskit import ClassicalRegister
from qiskit_aer import AerSimulator

# ============================================================
# PREPARE CIRCUIT FOR AER
# ============================================================

# qc_aer is your parameter-bound circuit
print("Original circuit operations:")
print(qc_aer.count_ops())

# Decompose high-level Qiskit objects such as ZZFeatureMap
qc_aer = qc_aer.decompose(reps=2)

print("\nAfter decomposition:")
print(qc_aer.count_ops())

# Verify that ZZFeatureMap is gone
assert "ZZFeatureMap" not in qc_aer.count_ops()

# ============================================================
# RUN AER
# ============================================================

result = simulator.run(
    qc_aer,
    shots=1024,
    seed_simulator=42
).result()

counts = result.get_counts()

print("\nCounts:")
print(counts)

while "ZZFeatureMap" in qc_aer.count_ops():
    qc_aer = qc_aer.decompose()

print(qc_aer.count_ops())

print(qc_aer.count_ops())
print("Parameters remaining:", len(qc_aer.parameters))

Original circuit operations:
OrderedDict({'u': 40, 'cx': 30, 'measure': 4})

After decomposition:
OrderedDict({'u': 40, 'cx': 30, 'measure': 4})

Counts:
{'1001': 191, '0011': 15, '1100': 38, '0010': 222, '0101': 99, '1000': 83, '1111': 14, '0111': 78, '1110': 112, '1101': 64, '1010': 32, '0001': 18, '0100': 24, '0110': 29, '0000': 3, '1011': 2}
OrderedDict({'u': 40, 'cx': 30, 'measure': 4})
OrderedDict({'u': 40, 'cx': 30, 'measure': 4})
Parameters remaining: 0


In [97]:
# ============================================================
# CALCULATE ZIII EXPECTATION FROM AER COUNTS
# ============================================================

shots = sum(counts.values())

z_expectation = 0.0

for bitstring, count in counts.items():
    # q0 is the RIGHTMOST bit in the displayed bitstring
    q0_bit = bitstring[-1]

    if q0_bit == "0":
        z_expectation += count
    else:
        z_expectation -= count

z_expectation /= shots

print("=" * 60)
print("AER ZIII EXPECTATION")
print("=" * 60)
print("Shots:", shots)
print("ZIII expectation:", z_expectation)

AER ZIII EXPECTATION
Shots: 1024
ZIII expectation: 0.060546875


In [98]:
classifier_weight = 2.050361
classifier_bias = 0.04361313

aer_logit = (
    classifier_weight * z_expectation
    + classifier_bias
)

aer_prediction = int(aer_logit >= 0)

print("=" * 60)
print("AER → CLASSIFIER")
print("=" * 60)
print("Z expectation:", z_expectation)
print("Logit:", aer_logit)
print("Prediction:", aer_prediction)

AER → CLASSIFIER
Z expectation: 0.060546875
Logit: 0.16775608117187502
Prediction: 1


In [99]:
print("=" * 60)
print("PER-QUBIT Z EXPECTATIONS FROM AER")
print("=" * 60)

shots = sum(counts.values())

for q in range(4):
    expectation = 0.0

    for bitstring, count in counts.items():
        # Qiskit classical bit ordering:
        # q0 -> rightmost bit
        bit = bitstring[-1 - q]

        if bit == "0":
            expectation += count
        else:
            expectation -= count

    expectation /= shots

    print(f"q{q} <Z> = {expectation:.9f}")

PER-QUBIT Z EXPECTATIONS FROM AER
q0 <Z> = 0.060546875
q1 <Z> = 0.015625000
q2 <Z> = 0.105468750
q3 <Z> = -0.046875000


In [101]:
from qiskit_aer import AerSimulator
import numpy as np

# Start from your already parameter-bound, decomposed circuit
qc_sv = qc_aer.remove_final_measurements(inplace=False)

# Explicitly request the statevector
qc_sv.save_statevector()

print("=" * 60)
print("STATEVECTOR CIRCUIT")
print("=" * 60)
print("Parameters remaining:", len(qc_sv.parameters))
print("Operations:", qc_sv.count_ops())

STATEVECTOR CIRCUIT
Parameters remaining: 0
Operations: OrderedDict({'u': 40, 'cx': 30, 'save_statevector': 1})


In [102]:
sv_simulator = AerSimulator(method="statevector")

result_sv = sv_simulator.run(
    qc_sv,
    seed_simulator=42
).result()

statevector = result_sv.get_statevector()

print("=" * 60)
print("STATEVECTOR SUCCESSFULLY CALCULATED")
print("=" * 60)
print("Number of amplitudes:", len(statevector))

STATEVECTOR SUCCESSFULLY CALCULATED
Number of amplitudes: 16


C:\Users\jneelam\AppData\Local\Temp\ipykernel_14116\490879280.py:13: DeprecationWarning: The return type of saved statevectors has been changed from a `numpy.ndarray` to a `qiskit.quantum_info.Statevector` as of qiskit-aer 0.10. Accessing numpy array attributes is deprecated and will result in an error in a future release. To continue using saved result objects as arrays you can explicitly cast them using  `np.asarray(object)`.
  print("Number of amplitudes:", len(statevector))


In [105]:
import numpy as np

# Convert Qiskit Statevector to NumPy array
sv_array = np.asarray(statevector)

# Probabilities
probs = np.abs(sv_array) ** 2

print("=" * 60)
print("EXACT AER STATEVECTOR Z EXPECTATIONS")
print("=" * 60)

for q in range(4):
    expectation = 0.0

    for basis_index, probability in enumerate(probs):
        bit = (basis_index >> q) & 1

        if bit == 0:
            expectation += probability
        else:
            expectation -= probability

    print(f"q{q} <Z> = {expectation:.12f}")

EXACT AER STATEVECTOR Z EXPECTATIONS
q0 <Z> = 0.051815262522
q1 <Z> = 0.037788935566
q2 <Z> = 0.088039025771
q3 <Z> = -0.065633615208


In [106]:
# ============================================================
# EXACT ZIII EXPECTATION
# ============================================================

ziii_expectation = 0.0

for basis_index, probability in enumerate(probs):

    # ZIII -> Z acting on q3 under Qiskit's Pauli convention
    bit = (basis_index >> 3) & 1

    if bit == 0:
        ziii_expectation += probability
    else:
        ziii_expectation -= probability

print("=" * 60)
print("EXACT ZIII EXPECTATION")
print("=" * 60)

print(f"ZIII = {ziii_expectation:.12f}")
print(f"Original EstimatorQNN = {-0.06087241:.12f}")
print(f"Difference = {ziii_expectation - (-0.06087241):.12f}")

EXACT ZIII EXPECTATION
ZIII = -0.065633615208
Original EstimatorQNN = -0.060872410000
Difference = -0.004761205208


In [107]:
# ============================================================
# CORRECT ZIII FROM MEASUREMENT COUNTS
# ============================================================

shots = sum(counts.values())

z_q3 = 0.0

for bitstring, count in counts.items():

    # Qiskit displays q3 on the LEFT
    q3_bit = bitstring[0]

    if q3_bit == "0":
        z_q3 += count
    else:
        z_q3 -= count

z_q3 /= shots

print("=" * 60)
print("CORRECT AER ZIII MEASUREMENT")
print("=" * 60)
print("Shots:", shots)
print("ZIII / q3 expectation:", z_q3)

# Trained classical classifier
classifier_weight = 2.050361
classifier_bias = 0.04361313

logit = classifier_weight * z_q3 + classifier_bias
prediction = int(logit >= 0)

print("Logit:", logit)
print("Prediction:", prediction)

CORRECT AER ZIII MEASUREMENT
Shots: 1024
ZIII / q3 expectation: -0.046875
Logit: -0.052497541875
Prediction: 0


In [108]:
# ----------------------------------------------------
# Process each sample
# ----------------------------------------------------

shots_total = sum(counts.values())

z_expectation = 0.0

for bitstring, count in counts.items():

    # Observable = ZIII
    # Therefore measure q3.
    # Qiskit count strings are displayed q3 q2 q1 q0.
    q3_bit = bitstring[0]

    if q3_bit == "0":
        z_expectation += count
    else:
        z_expectation -= count

z_expectation /= shots_total

# Classical trained classifier
logit = classifier_weight * z_expectation + classifier_bias

prediction = int(logit >= 0)

In [113]:
import numpy as np

# ============================================================
# FIND 04B TEST DATA
# ============================================================

def find_array(candidates):
    for name in candidates:
        if name in globals():
            obj = globals()[name]

            if isinstance(obj, np.ndarray):
                print(f"Found {name}: shape={obj.shape}")
                return np.asarray(obj)

    return None


X_test = find_array([
    "X_test_scaled_04B",
    "X_test_04B",
    "X_test_scaled",
    "X_test_pca_scaled",
    "X_test_pca",
])

y_test = find_array([
    "y_test_04B",
    "y_test",
    "y_test_pca",
])


if X_test is None:
    raise RuntimeError(
        "Could not find the 04B test feature array in memory."
    )

if y_test is None:
    raise RuntimeError(
        "Could not find the 04B test label array in memory."
    )


X_test = np.asarray(X_test, dtype=float)
y_test = np.asarray(y_test).astype(int)


print()
print("=" * 70)
print("04B DATA READY")
print("=" * 70)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("X_test dtype:", X_test.dtype)
print("y_test dtype:", y_test.dtype)

print()
print("First test sample:")
print(X_test[0])

print()
print("First test label:")
print(y_test[0])

Found X_test_scaled_04B: shape=(2956, 4)
Found y_test: shape=(2956,)

04B DATA READY
X_test shape: (2956, 4)
y_test shape: (2956,)
X_test dtype: float64
y_test dtype: int64

First test sample:
[ 0.92402527 -1.18636989  0.61522836  0.41028873]

First test label:
1


In [114]:
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

assert X_test.ndim == 2
assert X_test.shape[1] == 4
assert len(X_test) == len(y_test)

print("DATA CHECK PASSED")

X_test: (2956, 4)
y_test: (2956,)
DATA CHECK PASSED


In [112]:
# ============================================================
# EXPERIMENT 06A — SELF-CONTAINED AER VALIDATION
# ============================================================
#
# Purpose:
#   Reproduce the trained 04B quantum model using Aer and
#   compare IDEAL / DEPOLARIZING / READOUT / COMBINED noise.
#
# IMPORTANT:
#   This cell does NOT depend on previously defined:
#       feature_map
#       ansatz
#       quantum_model_04B
#       qnn_04B
#       model_04B
#
# ============================================================

import numpy as np
import pandas as pd
import time

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.quantum_info import SparsePauliOp

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
SHOTS = 1024

N_QUBITS = 4

print("=" * 70)
print("06A — SELF-CONTAINED AER VALIDATION")
print("=" * 70)

print("Qubits:", N_QUBITS)
print("Shots:", SHOTS)
print("Seed:", SEED)


# ============================================================
# 2. LOAD TEST DATA
# ============================================================

# Make sure the correct 04B data exists.
#
# If X_test_scaled_04B is already in memory, use it.
# Otherwise load it from your saved .npy file.

try:
    X_test = np.asarray(X_test_scaled_04B, dtype=float)
    y_test = np.asarray(y_test_04B, dtype=int)

    print("\nUsing existing X_test_scaled_04B / y_test_04B.")

except NameError:

    print("\n04B arrays not found in memory.")
    print("Trying to load them from disk...")

    X_test = np.load("../data/processed/X_test_scaled_04B.npy")
    y_test = np.load("../data/processed/y_test_04B.npy")

    X_test = np.asarray(X_test, dtype=float)
    y_test = np.asarray(y_test, dtype=int)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

assert X_test.shape[1] == 4
assert len(X_test) == len(y_test)


# ============================================================
# 3. TRAINED QUANTUM PARAMETERS FROM MODEL 04B
# ============================================================

quantum_weights = np.array([
     1.5234425,
     1.3247753,
    -0.30822906,
     1.6413008,
    -1.8245659,
     0.00345224,
    -0.04901591,
     1.5064764,
     0.8815429,
    -0.73362815,
     0.8691962,
     0.1010682
], dtype=float)


# Classical layer parameters
classifier_weight = 2.050361
classifier_bias = 0.04361313


print("\n" + "=" * 70)
print("TRAINED PARAMETERS")
print("=" * 70)

print("Quantum parameters:", len(quantum_weights))
print("Quantum weights:")
print(quantum_weights)

print("\nClassifier weight:", classifier_weight)
print("Classifier bias:", classifier_bias)


# ============================================================
# 4. RECREATE EXACT CIRCUIT ARCHITECTURE
# ============================================================

feature_map = ZZFeatureMap(
    feature_dimension=N_QUBITS,
    reps=1,
    entanglement="full"
)

ansatz = RealAmplitudes(
    num_qubits=N_QUBITS,
    reps=2,
    entanglement="full"
)

print("\n" + "=" * 70)
print("CIRCUIT ARCHITECTURE")
print("=" * 70)

print("Feature map:")
print(feature_map)

print("\nAnsatz:")
print(ansatz)

print("\nFeature-map parameters:")
print(feature_map.parameters)

print("\nAnsatz parameters:")
print(ansatz.parameters)


# ============================================================
# 5. BUILD FULL PARAMETERIZED CIRCUIT
# ============================================================

full_circuit = feature_map.compose(ansatz)

print("\n" + "=" * 70)
print("FULL PARAMETERIZED CIRCUIT")
print("=" * 70)

print(full_circuit)

print("\nNumber of circuit parameters:",
      len(full_circuit.parameters))


# ============================================================
# 6. CREATE OBSERVABLE
# ============================================================
#
# IMPORTANT:
#
# EstimatorQNN reported:
#
#     SparsePauliOp(['ZIII'])
#
# Therefore we use EXACTLY ZIII.
#
# Qiskit Pauli ordering is important:
# ZIII corresponds to the first Pauli character in the
# representation, while measurement bitstrings require
# careful interpretation.
#
# We will calculate the expectation directly from counts
# using q0, matching the observable used by EstimatorQNN.
# ============================================================

observable = SparsePauliOp("ZIII")

print("\n" + "=" * 70)
print("OBSERVABLE")
print("=" * 70)

print(observable)


# ============================================================
# 7. PARAMETER BINDING FUNCTION
# ============================================================

def create_bound_circuit(x, quantum_weights):
    """
    Create a fully bound circuit for one input sample.

    x:
        Four PCA features.

    quantum_weights:
        Twelve trained variational parameters.
    """

    x = np.asarray(x, dtype=float)
    quantum_weights = np.asarray(
        quantum_weights,
        dtype=float
    )

    if x.shape != (4,):
        raise ValueError(
            f"Expected x shape (4,), got {x.shape}"
        )

    if quantum_weights.shape != (12,):
        raise ValueError(
            f"Expected 12 quantum weights, "
            f"got {quantum_weights.shape}"
        )

    # IMPORTANT:
    # We bind parameters directly to the original circuit.
    # We do NOT mix parameter objects from different circuits.

    bind_dict = {}

    # Feature-map parameters
    feature_params = list(feature_map.parameters)

    # Sort them according to x[0], x[1], x[2], x[3]
    feature_params = sorted(
        feature_params,
        key=lambda p: str(p)
    )

    for param, value in zip(feature_params, x):
        bind_dict[param] = float(value)

    # Ansatz parameters
    ansatz_params = list(ansatz.parameters)

    ansatz_params = sorted(
        ansatz_params,
        key=lambda p: str(p)
    )

    for param, value in zip(
        ansatz_params,
        quantum_weights
    ):
        bind_dict[param] = float(value)

    bound = full_circuit.assign_parameters(
        bind_dict
    )

    return bound


# ============================================================
# 8. DECOMPOSE FOR AER
# ============================================================

def prepare_for_aer(bound_circuit):
    """
    Decompose high-level Qiskit circuit instructions
    into Aer-supported instructions.
    """

    qc = bound_circuit.decompose(
        reps=10
    )

    # Add measurements
    qc = qc.copy()

    qc.barrier()

    qc.measure_all()

    return qc


# ============================================================
# 9. TEST ONE CIRCUIT
# ============================================================

x_single = np.asarray(
    X_test[0],
    dtype=float
)

print("\n" + "=" * 70)
print("SINGLE SAMPLE")
print("=" * 70)

print("x =", x_single)
print("true label =", y_test[0])


bound_single = create_bound_circuit(
    x_single,
    quantum_weights
)

print("\nRemaining parameters:",
      len(bound_single.parameters))

assert len(bound_single.parameters) == 0


qc_single = prepare_for_aer(
    bound_single
)

print("\nAer operations:")
print(qc_single.count_ops())


# ============================================================
# 10. NOISE MODELS
# ============================================================

def create_ideal_noise_model():
    """
    Ideal = no noise.
    """
    return None


def create_depolarizing_noise_model():
    """
    Depolarizing noise model.

    NOTE:
    These probabilities should be reported explicitly in
    the thesis and kept fixed across all experiments.
    """

    noise_model = NoiseModel()

    # Example NISQ-level depolarizing probabilities.
    # Keep these consistent with your experimental design.
    p1 = 0.001
    p2 = 0.01

    error_1q = depolarizing_error(
        p1,
        1
    )

    error_2q = depolarizing_error(
        p2,
        2
    )

    noise_model.add_all_qubit_quantum_error(
        error_1q,
        ["u"]
    )

    noise_model.add_all_qubit_quantum_error(
        error_2q,
        ["cx"]
    )

    return noise_model


def create_readout_noise_model():
    """
    Symmetric readout error.
    """

    noise_model = NoiseModel()

    p01 = 0.02
    p10 = 0.02

    readout_error = ReadoutError([
        [1 - p01, p01],
        [p10, 1 - p10]
    ])

    noise_model.add_all_qubit_readout_error(
        readout_error
    )

    return noise_model


def create_combined_noise_model():
    """
    Depolarizing + readout noise.
    """

    noise_model = NoiseModel()

    # Quantum gate noise
    p1 = 0.001
    p2 = 0.01

    error_1q = depolarizing_error(
        p1,
        1
    )

    error_2q = depolarizing_error(
        p2,
        2
    )

    noise_model.add_all_qubit_quantum_error(
        error_1q,
        ["u"]
    )

    noise_model.add_all_qubit_quantum_error(
        error_2q,
        ["cx"]
    )

    # Readout noise
    p01 = 0.02
    p10 = 0.02

    readout_error = ReadoutError([
        [1 - p01, p01],
        [p10, 1 - p10]
    ])

    noise_model.add_all_qubit_readout_error(
        readout_error
    )

    return noise_model


# ============================================================
# 11. EXPECTATION FROM COUNTS
# ============================================================

def q0_z_expectation_from_counts(
    counts,
    shots
):
    """
    Calculate <Z> for q0.

    Qiskit count strings are displayed with the highest
    classical bit on the left.

    We explicitly reverse the string so that:
        bits[0] = q0
        bits[1] = q1
        bits[2] = q2
        bits[3] = q3

    Z expectation:
        +1 for |0>
        -1 for |1>
    """

    expectation = 0.0

    for bitstring, count in counts.items():

        bitstring = bitstring.replace(" ", "")

        # Reverse because Qiskit displays c3 c2 c1 c0
        bits = bitstring[::-1]

        q0 = int(bits[0])

        if q0 == 0:
            eigenvalue = +1
        else:
            eigenvalue = -1

        expectation += (
            eigenvalue * count / shots
        )

    return expectation


# ============================================================
# 12. CLASSIFIER
# ============================================================

def expectation_to_prediction(
    z_expectation
):
    """
    Convert quantum expectation to the trained
    classical classifier output.

    HybridClassifier:
        logit = weight * quantum_output + bias

    Prediction:
        logit >= 0 -> class 1
        logit < 0  -> class 0
    """

    logit = (
        classifier_weight * z_expectation
        + classifier_bias
    )

    prediction = int(logit >= 0)

    return logit, prediction


# ============================================================
# 13. SINGLE SAMPLE VALIDATION
# ============================================================

def run_single_aer_sample(
    simulator,
    x,
    noise_model,
    label
):

    bound = create_bound_circuit(
        x,
        quantum_weights
    )

    qc = prepare_for_aer(
        bound
    )

    start = time.time()

    result = simulator.run(
        qc,
        shots=SHOTS,
        seed_simulator=SEED,
        noise_model=noise_model
    ).result()

    runtime = time.time() - start

    counts = result.get_counts()

    z_exp = q0_z_expectation_from_counts(
        counts,
        SHOTS
    )

    logit, prediction = expectation_to_prediction(
        z_exp
    )

    print("\n" + "=" * 60)
    print(label)
    print("=" * 60)

    print("Z expectation:", z_exp)
    print("Logit:", logit)
    print("Prediction:", prediction)
    print("True label:", y_test[0])
    print("Runtime:", runtime, "seconds")

    return {
        "condition": label,
        "z_expectation": z_exp,
        "logit": logit,
        "prediction": prediction,
        "true_label": int(y_test[0]),
        "counts": counts
    }


# ============================================================
# 14. CREATE SIMULATORS
# ============================================================

simulator = AerSimulator()


noise_models = {
    "ideal": create_ideal_noise_model(),
    "depolarizing": create_depolarizing_noise_model(),
    "readout": create_readout_noise_model(),
    "combined": create_combined_noise_model()
}


# ============================================================
# 15. SINGLE-SAMPLE TEST
# ============================================================

single_results = []

for condition, noise_model in noise_models.items():

    result = run_single_aer_sample(
        simulator=simulator,
        x=x_single,
        noise_model=noise_model,
        label=condition
    )

    single_results.append(result)


# ============================================================
# 16. FULL DATASET EVALUATION
# ============================================================

def evaluate_condition(
    condition,
    noise_model,
    X,
    y
):

    print("\n")
    print("=" * 70)
    print(f"RUNNING FULL 06A — {condition.upper()}")
    print("=" * 70)

    predictions = []

    start = time.time()

    for i, x in enumerate(X):

        bound = create_bound_circuit(
            x,
            quantum_weights
        )

        qc = prepare_for_aer(
            bound
        )

        result = simulator.run(
            qc,
            shots=SHOTS,
            seed_simulator=SEED,
            noise_model=noise_model
        ).result()

        counts = result.get_counts()

        z_exp = q0_z_expectation_from_counts(
            counts,
            SHOTS
        )

        logit, prediction = expectation_to_prediction(
            z_exp
        )

        predictions.append(prediction)

        if (
            i < 2
            or (i + 1) % 100 == 0
            or (i + 1) == len(X)
        ):
            print(
                f"{condition}: "
                f"{i + 1}/{len(X)}"
            )

    runtime = time.time() - start

    predictions = np.asarray(
        predictions,
        dtype=int
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = np.mean(
        predictions == y
    )

    tp = np.sum(
        (predictions == 1) &
        (y == 1)
    )

    fp = np.sum(
        (predictions == 1) &
        (y == 0)
    )

    fn = np.sum(
        (predictions == 0) &
        (y == 1)
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    print("\n" + "=" * 70)
    print(f"{condition.upper()} COMPLETE")
    print("=" * 70)

    print(f"Accuracy:  {accuracy:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall:    {recall:.6f}")
    print(f"F1:        {f1:.6f}")
    print(
        f"Runtime:   {runtime / 60:.2f} minutes"
    )

    return {
        "condition": condition,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "runtime_seconds": runtime,
        "predictions": predictions
    }


# ============================================================
# 17. RUN ALL FOUR CONDITIONS
# ============================================================

all_results = []

for condition, noise_model in noise_models.items():

    result = evaluate_condition(
        condition=condition,
        noise_model=noise_model,
        X=X_test,
        y=y_test
    )

    all_results.append(result)


# ============================================================
# 18. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame([
    {
        "condition": r["condition"],
        "accuracy": r["accuracy"],
        "precision": r["precision"],
        "recall": r["recall"],
        "f1": r["f1"],
        "runtime_seconds": r["runtime_seconds"]
    }
    for r in all_results
])

print("\n")
print("=" * 70)
print("06A RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 19. ACCURACY DEGRADATION
# ============================================================

ideal_accuracy = results_df.loc[
    results_df["condition"] == "ideal",
    "accuracy"
].iloc[0]


results_df["accuracy_drop_vs_ideal"] = (
    ideal_accuracy -
    results_df["accuracy"]
)

results_df[
    "relative_accuracy_drop_percent"
] = (
    results_df["accuracy_drop_vs_ideal"]
    / ideal_accuracy
) * 100


print("\n")
print("=" * 70)
print("06A ACCURACY DEGRADATION")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 20. PREDICTION DISTRIBUTIONS
# ============================================================

print("\n")
print("=" * 70)
print("PREDICTION DISTRIBUTIONS")
print("=" * 70)

for result in all_results:

    condition = result["condition"]
    predictions = result["predictions"]

    unique, counts = np.unique(
        predictions,
        return_counts=True
    )

    distribution = dict(
        zip(unique, counts)
    )

    print(
        f"\n{condition}:"
    )

    print(
        "Predictions:",
        distribution
    )


true_unique, true_counts = np.unique(
    y_test,
    return_counts=True
)

print("\nTrue labels:")
print(
    dict(
        zip(
            true_unique,
            true_counts
        )
    )
)


# ============================================================
# 21. SAVE RESULTS
# ============================================================

results_to_save = results_df.drop(
    columns=["predictions"],
    errors="ignore"
)

results_to_save.to_csv(
    "../results/06A_aer_noise_results.csv",
    index=False
)

print("\nResults saved to:")
print(
    "../results/06A_aer_noise_results.csv"
)


# ============================================================
# 22. SAVE PREDICTIONS
# ============================================================

for result in all_results:

    condition = result["condition"]

    np.save(
        f"../results/06A_predictions_{condition}.npy",
        result["predictions"]
    )

print("Predictions saved.")


print("\n")
print("=" * 70)
print("06A COMPLETE")
print("=" * 70)

06A — SELF-CONTAINED AER VALIDATION
Qubits: 4
Shots: 1024
Seed: 42

04B arrays not found in memory.
Trying to load them from disk...


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/X_test_scaled_04B.npy'